# SmolVLA Final Training v9 (Colab)

Train SmolVLA with HPO-optimized hyperparameters on **cleaned** dataset.

- **Dataset:** `AdithyaRajendran/so101_grab_brain_t2` (239 episodes, 99845 frames)
  - Episodes 69 & 240 deleted (bad demos)
- **Task:** *"Grab the grey brain toy and place it inside the green container"*
- **Base model:** `lerobot/smolvla_base`
- **Push to:** `AdithyaRajendran/smolvla_so101_grab_brain_t2_v9`

### Key HPO findings (v3, 30 trials, 15 completed, 15 pruned)
- **LR = 4.2e-5** (was 1e-4 in all previous models — 2.4x lower!)
- **weight_decay = 8.5e-5** (was 1e-10 — much more regularization)
- **chunk_size = 10, n_action_steps = 10** (confirmed optimal)
- **freeze_vision = True** (frozen won, unfrozen was close at 0.02317 vs 0.02300)
- **Strong camera shift augmentation**: affine_degrees=10, translate=0.11
- **High contrast jitter (0.4)**, low brightness jitter (0.12)

### Action velocity stats (cleaned dataset, 117 eps sampled)
| Joint | σ(Δa) | \|Δ\|max |
|-------|--------|----------|
| shoulder_pan | 0.0078 | 0.1251 |
| shoulder_lift | 0.0098 | 0.1053 |
| elbow_flex | 0.0080 | 0.1111 |
| wrist_flex | 0.0091 | 0.1020 |
| wrist_roll | 0.0068 | 0.1315 |
| gripper | 0.0267 | 0.2937 |

Gripper ~3x higher variance is expected (binary open/close). MEAN_STD normalization handles this.

### Recommended runtime: **L4 or A100**
- ~2-3 hrs on A100, ~3-4 hrs on L4

In [7]:
# Check GPU
import torch, platform

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / 1024**3, 2))
else:
    raise RuntimeError("Please switch Colab runtime to GPU")

Python: 3.12.12
Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM (GB): 39.49


In [8]:
%%bash
# Install LeRobot
cd /content

if [ ! -d lerobot ]; then
  git clone https://github.com/huggingface/lerobot.git
fi

cd /content/lerobot

apt-get -qq update
apt-get -qq install -y ffmpeg

pip install -q "huggingface-hub[cli,hf-transfer]>=0.34.2,<0.36.0" "setuptools>=71.0.0,<81.0.0" "wandb>=0.24.0,<0.25.0"
pip install -q -e ".[smolvla]"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [9]:
%%bash
# Authenticate with Hugging Face (replace with your token)
huggingface-cli login --token YOUR_HF_TOKEN_HERE

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `Pi0.5` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Pi0.5`


In [10]:
# Enable W&B for experiment tracking (replace with your key)
import wandb
wandb.login(key="YOUR_WANDB_KEY_HERE")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [ ]:
# ============================================================
# CONFIGURATION - Best params from Optuna HPO v3
# (30 trials, best trial #11, avg final loss = 0.02300)
# Updated for cleaned dataset (239 eps, 99845 frames)
# ============================================================

import os
from pathlib import Path

# Dataset
DATASET_REPO_ID = "AdithyaRajendran/so101_grab_brain_t2"
NUM_FRAMES = 99845  # After deleting episodes 69 & 240

# Model
POLICY_PATH = "lerobot/smolvla_base"
POLICY_REPO_ID = "AdithyaRajendran/smolvla_so101_grab_brain_t2_v9"
JOB_NAME = "smolvla_so101_grab_brain_t2_v9"

# === BEST PARAMS FROM OPTUNA HPO v3 (trial #11) ===
LEARNING_RATE = 4.24e-5
WEIGHT_DECAY = 8.49e-5
CHUNK_SIZE = 10
N_ACTION_STEPS = 10
AFFINE_DEGREES = 9.94
AFFINE_TRANSLATE = 0.1072
COLOR_JITTER_BRIGHTNESS = 0.119
COLOR_JITTER_CONTRAST = 0.399

# Training settings
# NOTE: Use lowercase "true"/"false" strings — draccus rejects Python's "True"/"False"
FREEZE_VISION_ENCODER = "true"   # HPO confirmed frozen is best
TRAIN_EXPERT_ONLY = "true"       # Frozen vision → expert only
BATCH_SIZE = 32                  # A100/L4 can handle this with frozen vision

# 6.5 epochs (known optimal from previous experiments)
STEPS = int(6.5 * NUM_FRAMES / BATCH_SIZE)

# Scheduler
WARMUP_STEPS = 1000
DECAY_LR = LEARNING_RATE * 0.025  # Decay to 2.5% of peak LR

OUTPUT_DIR = f"/content/outputs/{JOB_NAME}"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Steps: {STEPS} (6.5 epochs with batch_size={BATCH_SIZE})")
print(f"Output: {OUTPUT_DIR}")
print(f"LR: {LEARNING_RATE}, WD: {WEIGHT_DECAY}")
print(f"Chunk: {CHUNK_SIZE}, N_action: {N_ACTION_STEPS}")
print(f"Freeze vision: {FREEZE_VISION_ENCODER}, Expert only: {TRAIN_EXPERT_ONLY}")
print(f"Augmentation: affine_deg={AFFINE_DEGREES}, translate={AFFINE_TRANSLATE}")
print(f"  brightness={COLOR_JITTER_BRIGHTNESS}, contrast={COLOR_JITTER_CONTRAST}")

In [12]:
%%bash -s "$OUTPUT_DIR" "$DATASET_REPO_ID" "$POLICY_PATH" "$POLICY_REPO_ID" "$JOB_NAME" "$LEARNING_RATE" "$WEIGHT_DECAY" "$CHUNK_SIZE" "$N_ACTION_STEPS" "$BATCH_SIZE" "$STEPS" "$WARMUP_STEPS" "$DECAY_LR" "$FREEZE_VISION_ENCODER" "$TRAIN_EXPERT_ONLY" "$AFFINE_DEGREES" "$AFFINE_TRANSLATE" "$COLOR_JITTER_BRIGHTNESS" "$COLOR_JITTER_CONTRAST"

cd /content/lerobot

# Clean output dir from previous failed runs
rm -rf "$1"

# Build image transforms JSON (draccus cannot parse nested tfs.X.Y.Z args)
BRIGHT_LO=$(python3 -c "print(1.0-${18})")
BRIGHT_HI=$(python3 -c "print(1.0+${18})")
CONTR_LO=$(python3 -c "print(1.0-${19})")
CONTR_HI=$(python3 -c "print(1.0+${19})")

TFS_JSON="{\"brightness\": {\"type\": \"ColorJitter\", \"kwargs\": {\"brightness\": [${BRIGHT_LO}, ${BRIGHT_HI}]}}, \"contrast\": {\"type\": \"ColorJitter\", \"kwargs\": {\"contrast\": [${CONTR_LO}, ${CONTR_HI}]}}, \"saturation\": {\"type\": \"ColorJitter\", \"kwargs\": {\"saturation\": [0.5, 1.5]}}, \"hue\": {\"type\": \"ColorJitter\", \"kwargs\": {\"hue\": [-0.05, 0.05]}}, \"sharpness\": {\"type\": \"SharpnessJitter\", \"kwargs\": {\"sharpness\": [0.5, 1.5]}}, \"affine\": {\"type\": \"RandomAffine\", \"kwargs\": {\"degrees\": [-${16}, ${16}], \"translate\": [${17}, ${17}]}}}"

python src/lerobot/scripts/lerobot_train.py \
    --dataset.repo_id="$2" \
    --output_dir="$1" \
    --job_name="$5" \
    --policy.path="$3" \
    --policy.repo_id="$4" \
    --policy.push_to_hub=true \
    --policy.chunk_size="$8" \
    --policy.n_action_steps="$9" \
    --policy.optimizer_lr="$6" \
    --policy.optimizer_weight_decay="$7" \
    --policy.freeze_vision_encoder="${14}" \
    --policy.train_expert_only="${15}" \
    --policy.scheduler_warmup_steps="${12}" \
    --policy.scheduler_decay_steps="${11}" \
    --policy.scheduler_decay_lr="${13}" \
    --batch_size="${10}" \
    --steps="${11}" \
    --log_freq=100 \
    --save_freq=2000 \
    --num_workers=4 \
    '--policy.normalization_mapping={"ACTION": "MEAN_STD", "STATE": "MEAN_STD", "VISUAL": "IDENTITY"}' \
    '--rename_map={"observation.images.front": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}' \
    --dataset.image_transforms.enable=true \
    --dataset.image_transforms.max_num_transforms=3 \
    --dataset.image_transforms.random_order=false \
    "--dataset.image_transforms.tfs=${TFS_JSON}" \
    --wandb.enable=true

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Reducing the number of VLM layers to 16 ...


2026-03-04 03:06:37.681337: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 03:06:37.699876: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772593597.722036    3593 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772593597.729341    3593 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772593597.748270    3593 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [13]:
# Quick listing of saved outputs
import os
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for f in files[:20]:
        print(f"{subindent}{f}")
    if level >= 2:
        dirs[:] = []

smolvla_so101_grab_brain_t2_v8/
  wandb/
    debug-internal.log
    debug.log
    run-20260304_030650-49dg6rhq/
      run-49dg6rhq.wandb
  checkpoints/
    018000/
    016000/
    020000/
    008000/
    020481/
    012000/
    006000/
    002000/
    010000/
    004000/
    014000/


## After Training

The model is automatically pushed to `AdithyaRajendran/smolvla_so101_grab_brain_t2_v9`.

### Local eval command
Run this on your local machine with the robot:
```bash
cd ~/lerobot
bash run_smolvla_eval.sh
```
(Update `POLICY_REPO` in `run_smolvla_eval.sh` to point to `v9` if needed)

### If OOM during training
1. Reduce `BATCH_SIZE` to 16
2. If still OOM, reduce `BATCH_SIZE` to 8
3. Vision is already frozen, so OOM is unlikely with batch_size=32